In [1]:
import warnings
warnings.simplefilter(action='ignore')

import argparse
import os,cv2,math
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import argparse
import os
import random,numpy
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
from torchvision.models.feature_extraction import create_feature_extractor
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

Random Seed:  999


In [4]:
workers = 2
batch_size = 50
nz = 100
num_epochs = 5
lr = 0.0001
beta1 = 0.5
ngpu=1
ngf,nc = 3,3
ndf = 64

transform = transforms.Compose([
    transforms.Resize(200),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
fsc_submission=pandas.read_csv("/kaggle/input/cidaut-ai-fake-scene-classification-2024/sample_submission.csv", index_col ="image")#.to_dict('list')
train = torchvision.datasets.ImageFolder(root=f'/kaggle/input/fake-scene-dataset/train',transform=transform)

dataloader = torch.utils.data.DataLoader(train,batch_size=batch_size,shuffle=True,num_workers=2)

#print(len(dataloader),len(testloader))
device = torch.device("cuda" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

In [5]:
class EffnetModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        effnet = torchvision.models.efficientnet_v2_m(weights=torchvision.models.efficientnet.EfficientNet_V2_M_Weights.DEFAULT)
        self.model = create_feature_extractor(effnet, ['flatten'])
        self.nn_fracture = torch.nn.Sequential(
            torch.nn.Linear(1280, 1),
            torch.nn.Sigmoid()
        )
    def forward(self, x):
        x = self.model(x)['flatten']
        x = self.nn_fracture(x)
        return x

In [6]:
EFF_NET = EffnetModel().float()
EFF_NET= nn.DataParallel(EFF_NET).to(device)
#EFF_NET
#EFF_NET.load_state_dict(torch.load(f"/kaggle/input/fsc-dataset/8.072827768046409e-05 88.0.pth",weights_only=False,map_location=torch.device('cpu')))

Downloading: "https://download.pytorch.org/models/efficientnet_v2_m-dc08266a.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_m-dc08266a.pth
100%|██████████| 208M/208M [00:01<00:00, 195MB/s]


In [7]:
criterion = nn.BCEWithLogitsLoss()

# CrossEntropyLoss
# MSELoss
# L1Loss
# BCELoss
# BCEWithLogitsLoss

optimizer = optim.AdamW(EFF_NET.parameters(), lr=lr, betas=(beta1, 0.999))
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.86)

In [8]:
print(dataloader.dataset.classes)
z_w_=[]
high_rf,i_w,z_k=0,0,0
high_rf_=1

lab={
    'editada' : 0,
    'real' : 1
}
while True:
    z_,z,z_w=0,0,{}
    correct=0
    for i, data in enumerate(dataloader, 0):
        label = data[1].to(device).float()
        raw_output = EFF_NET(data[0].to(device)).float().view(-1)
        err_real = criterion(raw_output, label)
        optimizer.zero_grad()
        err_real.backward()
        optimizer.step()
        
    z_w_.append(err_real.item())
    print(f"EPOCH: {z_k} LOSS_FSC: {err_real.item()}")
    raw_output = raw_output.cpu().detach().numpy()
    
    print(r2_score(raw_output, label.cpu().detach().numpy()))
    #print(z_)	
        
    if len(z_w_)>=3:
        #if len([True for i in range(1,4) if z_w_[len(z_w_)-i]<=z_w_[len(z_w_)-3] and z_w_[len(z_w_)-i]>=z_w_[len(z_w_)-4]])==3:
            z_w_=[]
            print(optimizer.param_groups[0]["lr"])
            scheduler.step()
            print(optimizer.param_groups[0]["lr"])
    
    if err_real.item()<=0.9:
        torch.save(EFF_NET.state_dict(),f'/kaggle/working/{err_real.item()} {z_k}.pth')# {z_add}{acc}
    
    if z_k==10:
        break
    z_k+=1
    i_w+=1
    

['editada', 'real']
EPOCH: 0 LOSS_FSC: 0.6882762312889099
-38.95582670647474
EPOCH: 1 LOSS_FSC: 0.606463611125946
-0.2530064383690005
EPOCH: 2 LOSS_FSC: 0.595558226108551
-0.051270765510345484
0.0001
8.6e-05
EPOCH: 3 LOSS_FSC: 0.5382628440856934
0.40605615598946887
EPOCH: 4 LOSS_FSC: 0.6166815161705017
-2.382126946164497
EPOCH: 5 LOSS_FSC: 0.5971721410751343
0.04194446694025289
8.6e-05
7.396e-05
EPOCH: 6 LOSS_FSC: 0.5568212866783142
0.685482424469969
EPOCH: 7 LOSS_FSC: 0.7420680522918701
-0.13234506791696954
EPOCH: 8 LOSS_FSC: 0.5329623818397522
0.9439415275112656
7.396e-05
6.36056e-05
EPOCH: 9 LOSS_FSC: 0.5466275811195374
0.45176596713668327
EPOCH: 10 LOSS_FSC: 0.5061484575271606
0.7140258994849624
